# CSE475 - Phase 1 baseline (Kaggle T4)

Trains `efficientnet_b0` on the starter dataset (acne/rosacea/normal).

Uses the portable harness `multi-source-skin-disease-fusion`:
- dataset downloaded from HF (`Neperl/skin-disease-acne-rosacea-normal`)
- checkpoint synced to HF (`Nirob-jon/cse475-skin-checkpoints`) after every epoch
- resumes from the latest HF checkpoint if this run was interrupted

**Setup required:** Add a write-scope HF token as a notebook **Secret** named `HF_TOKEN` (right panel -> Secrets -> Add secret).

In [ ]:
import os, getpass
from huggingface_hub import login, snapshot_download

HF_REPO = "Nirob-jon/cse475-skin-checkpoints"

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata  # not on Kaggle but harmless to try
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass.getpass("Paste your HF token (hf_...): ")
assert token, "No HF_TOKEN found"
login(token=token, add_to_git_credential=False)
print("logged in OK")

In [ ]:
# clone the harness (public repo)
!git clone --depth 1 https://github.com/nirjon001/multi-source-skin-disease-fusion.git
%cd multi-source-skin-disease-fusion

In [ ]:
# dataset (this machine has torch already, keep installs light)
DATA = snapshot_download("Neperl/skin-disease-acne-rosacea-normal", repo_type="dataset")
print("dataset at:", DATA)
!ls "$DATA"/train

In [ ]:
# light deps only (torch/torchvision preinstalled on Kaggle)
!pip install -q timm tqdm pyyaml imagehash scikit-learn huggingface_hub

In [ ]:
!python src/audit.py --data "$DATA" --check-corrupt --out results/audit_starter.json

In [ ]:
import matplotlib.pyplot as plt
import os, glob
from PIL import Image

plt.figure(figsize=(12, 4))
files = sorted(glob.glob(os.path.join(DATA, "train", "*", "*")))
for i, cls in enumerate(["Acne", "Normal", "Rosacea"]):
    c = [f for f in files if os.sep + cls + os.sep in f]
    if not c:
        c = sorted(glob.glob(os.path.join(DATA, "train", cls, "*")))
    plt.subplot(1, 3, i + 1)
    plt.imshow(Image.open(c[0]))
    plt.title(cls)
    plt.axis("off")
plt.show()

In [ ]:
# TRAIN (Kaggle profile: preinstalled CUDA torch, batch 32, AMP on)
!python src/train_resumable.py --config configs/baseline.yaml --data "$DATA" \
    --profile kaggle_t4 --hub hf --resume auto --hf-repo "$HF_REPO" --epochs 15

In [ ]:
import json, pathlib
r = json.load(open("results/phase1_baseline.json"))
print("test_acc   =", r["test_acc"])
print("macro_f1   =", r["macro_f1"])
print(r["confusion_matrix"])

## Copy results home

The checkpoint `phase1_baseline_last.pt` was already uploaded to HF after every epoch.
Download `results/phase1_baseline.json` in the output panel if you want to keep it (or just re-fetch from the next machine).